# CatBoost + 1-Minute Bars + Call Option Execution

This notebook runs the live mean-reversion engine with:

- underlying signal data from stock bars
- completed `1min` signal bars
- CatBoost entry filter enabled
- call option orders through IBKR

The signal remains based on the underlying stock. Only the execution instrument changes to a call option.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, Stock

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import (
    active_orders,
    all_orders,
    build_execution_instrument,
    filled_orders,
    print_order_snapshot,
    recent_fills,
    strategy_open_trades,
)
from model_filters.catboost_live_model_filter import CatBoostLiveModelFilter
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


## 1. User Settings

Set the underlying symbol and the call contract. Expiry is IBKR format `YYYYMMDD`.

In [ ]:
SYMBOL = "META"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 102

CALL_EXPIRY = "20260619"
CALL_STRIKE = 500.0
CALL_RIGHT = "C"

CATBOOST_MODEL_PATH = "models/live/catboost/meta_catboost.cbm"
CATBOOST_PROB_THRESHOLD = 0.55

OPEN_TRADES_PATH = f"{SYMBOL}_calls_1min_catboost_open_trades.csv"


## 2. Build Config

The signal bar config is still `1min`. The execution instrument is now `option`, so orders route to the call while entries and exits are still decided from the underlying stock.

In [ ]:
raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["paths"]["open_trades_path"] = OPEN_TRADES_PATH
raw["paths"]["catboost_model_path"] = CATBOOST_MODEL_PATH

raw["features"]["bar"] = {
    "type": "time",
    "timeframe": "1min",
    "history_window": 390,
}

raw["model"]["enabled"] = True
raw["model"]["prob_threshold"] = CATBOOST_PROB_THRESHOLD

raw["execution"]["instrument"] = {
    "type": "option",
    "exchange": "SMART",
    "currency": "USD",
    "expiry": CALL_EXPIRY,
    "strike": CALL_STRIKE,
    "right": CALL_RIGHT,
    "multiplier": 100.0,
    "limit_entry_offset_pct": 0.02,
    "limit_exit_offset_pct": 0.02,
}

# Optional: require the option itself to be down enough before double-down.
raw["double_down"]["price_rules"] = [
    {"basis": "underlying", "mode": "pct", "max_change": -0.01},
    {"basis": "execution", "mode": "pct", "max_change": -0.20},
]

config = LiveTradingConfig.from_dict(raw)
config.raw["features"]["bar"], config.raw["execution"]["instrument"]


## 3. Connect IBKR and Subscribe to Underlying Stock Bars

In [ ]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock = Stock(SYMBOL, "SMART", "USD")
ib.qualifyContracts(stock)

real_time_bars = ib.reqRealTimeBars(
    stock,
    barSize=5,
    whatToShow="TRADES",
    useRTH=False,
)

print("Connected:", ib.isConnected())
print("Signal underlying:", stock)


## 4. Build Strategy

`order_router` still uses IBKR. `execution_instrument.resolve_contract()` will qualify the call option when the first order path needs it.

In [ ]:
model_filter = CatBoostLiveModelFilter(
    model_path=config.paths["catboost_model_path"],
    feature_cols=config.model["feature_cols"],
    prob_threshold=config.model.get("prob_threshold", 0.50),
    enabled=config.model.get("enabled", True),
)

execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

order_router = IBKRLimitOrderRouter(ib=ib)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
    execution_instrument=execution_instrument,
)

print("Execution instrument:", execution_instrument.instrument_type)
print("Loaded strategy open trades:", len(algo.open_trades))


## 5. Inspect the Option Contract Before Running

This qualifies the option contract now so you can catch expiry/strike/right mistakes before attaching the live callback.

In [ ]:
option_contract = execution_instrument.resolve_contract()
option_contract


## 6. Start / Stop Callback

In [ ]:
real_time_bars.updateEvent += algo.on_bar
print("Attached algo.on_bar")


In [ ]:
real_time_bars.updateEvent -= algo.on_bar
print("Detached algo.on_bar")


## 7. Order and Trade Views

In [ ]:
print_order_snapshot(ib, algo)


In [ ]:
active_orders(ib)


In [ ]:
filled_orders(ib)


In [ ]:
recent_fills(ib)


In [ ]:
strategy_open_trades(algo)


In [ ]:
all_orders(ib)
